# DS-Harness starter

**Build an AI system around a small language model.**

Each task is an objective in English, sometimes with CSV files, and its answer is **one number**:
a computation stated in the text, a statistic over messy tables, or a regression fitted on a
training file. A model of 1.5 billion parameters asked directly gets the files wrong, the
arithmetic wrong and the format wrong. Your project is the system around it:

- how the objective and its files are read;
- how the system decides what kind of task it faces;
- what reaches the model;
- which tools exist and how the model calls them;
- how the answer is parsed and checked;
- how the 40 seconds of each call are spent.

**An efficient system matters more than the model.** On `dev.json`, the same 1.5B model goes from
10.6 / 100 (asked directly, answer well parsed) to 46.1 with a system that reads the files
itself, shows the model a clean view of them and runs the code it writes.

This notebook runs on a free Colab **T4 GPU**: the data, the first solution in detail, a tool and
a loop as a skeleton, the directions, how to measure, how to submit. Challenge page:
[ml-arena.com/viewchallenge/194](https://ml-arena.com/viewchallenge/194).

---

## 0. Setup

**Runtime → Change runtime type → T4 GPU.** The ML-Arena client is the package **`mlarena-sdk`**
(imported as `mlarena`; `pip install mlarena` is an unrelated package). Colab already has torch,
transformers, pandas and numpy.

The code is the public repository [racousin/ds-harness](https://github.com/racousin/ds-harness):

| file | what it is |
|---|---|
| `scoring.py` | the scorer and the delivery loop (`run_agent`) the leaderboard runs |
| `schema.md` | formats, score, timing, the machine: read it once |
| `localtest.py` | runs an agent file on `dev.json` the way the leaderboard does |
| `dsh.py` | the kit: model loading, answer parsers, calculator, tool-call parser, code runner |
| `stage1_direct.py` | the first solution (section 2) |
| `stage2_tool_loop.py` | a tool and a loop, with the loop body to write (section 3) |

In [1]:
!pip install -q mlarena-sdk accelerate
!git clone -q https://github.com/racousin/ds-harness
import os
os.chdir("ds-harness")
print(sorted(os.listdir(".")))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 6.3 MB/s eta 0:00:00
['.git', '.gitignore', 'README.md', 'dsh.py', 'env.py', 'hf_models.json', 'localtest.py', 'requirements.txt', 'schema.md', 'scoring.py', 'stage1_direct.py', 'stage2_tool_loop.py', 'test_dsh.py']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Your API key is on your ML-Arena **Profile** page (`mlk_user_...`). `download_dataset` writes
`dev.json`: 180 public tasks with their files, answers and types.

In [3]:
from getpass import getpass
import mlarena
from google.colab import userdata

CHALLENGE_ID = 194
api_key_mlarena = userdata.get('api_key_mlarena')

client = mlarena.connect(api_key=api_key_mlarena)
print(client.download_dataset(CHALLENGE_ID, "."))

['./dev.json']


In [4]:
import collections, json, re, tempfile, time
import pandas as pd
import dsh, scoring

dev = json.load(open("dev.json"))
print(len(dev), "tasks")

180 tasks


---

## 1. The data

### What your agent receives

Your `Agent.solve(tasks)` is called with 8 tasks at a time. A task is **only** its `id`, its
`objective` and the **paths** of its files. The platform writes the files of a call to disk just
before the call and deletes them after. `scoring.write_files` does exactly that here:

In [5]:
no_file = next(t for t in dev if not t["files"])
with_files = next(t for t in dev if t["type"] == "table.filter")
root = tempfile.mkdtemp()
payload = scoring.write_files([no_file, with_files], root)
for p in payload:
    print(json.dumps(p, indent=1), "\n")

{
 "id": "d_008",
 "objective": "Compute 88 \u00d7 32. The answer is an integer.",
 "files": []
} 

{
 "id": "d_001",
 "objective": "You are given daily electricity readings of a company's sites in meter.csv, with a data dictionary. What is the maximum outside temperature of the readings of the Grenoble lab site (in \u00b0C)? meter.csv uses ';' to separate columns and a comma as the decimal mark. Give the answer rounded to one decimal place.",
 "files": [
  "/tmp/tmpjaifw45_/d_001/meter.csv",
  "/tmp/tmpjaifw45_/d_001/data_dictionary.csv"
 ]
} 



In [7]:
for path in payload[1]["files"]:
    print(f"--- {os.path.basename(path)}")
    print("".join(open(path).readlines()[:8]))

--- meter.csv
read_no;loc;date;e_kwh;temp_ext;HEADS;TAR
1;S02;2024-11-10;454,1;8,1;130;flat
2;S03;2023-01-01;454,2;2,6;129;offpeak
3;S02;2024-06-05;267,5;18,8;90;peak
4;S03;2024-03-17;282,5;11,2;88;flat
5;S01;2023-05-10;114,5;12,3;4;peak
6;S04;2023-08-23;380,2;21,8;139;flat
7;S02;2023-08-13;180,6;20,9;51;peak

--- data_dictionary.csv
file,column,description,unit
meter.csv,read_no,"Reading number",
meter.csv,loc,"Site, coded (S01 Bordeaux office, S02 Grenoble lab, S03 Lille warehouse, S04 Nantes office, S05 Toulouse workshop)",
meter.csv,date,"Day of the reading",
meter.csv,e_kwh,"Electricity used over the day",kWh
meter.csv,temp_ext,"Mean outside temperature",°C
meter.csv,HEADS,"People present on site",people
meter.csv,TAR,"Tariff applied (peak, offpeak, flat)",



Read the objective of the second task again, then the files. The headers are short codes that
change from task to task; the objective names a column by its **description** in
`data_dictionary.csv`. The objective also states how to read the file: the separator, the decimal
mark, how missing values are written, values that carry their unit, duplicated rows, a value
that means *not recorded*. Getting all of this right is the system's job, not the model's.

### What only you see

`dev.json` also carries, for each task, its `answer`, its tolerance `tol` and its `type`. The
agent never gets them: your system must recognise the kind of task from the objective.

In [8]:
df = pd.DataFrame([{"type": t["type"], "family": t["type"].split(".")[0], "files": len(t["files"])}
                   for t in dev])
df.groupby(["family", "type"]).size().rename("tasks").reset_index()

,family,type,tasks
0,calc,calc.arithmetic,8
1,calc,calc.dates,8
2,calc,calc.functions,7
3,calc,calc.growth,7
4,calc,calc.integer,7
5,calc,calc.percent,7
6,calc,calc.probability,7
7,calc,calc.stats_inline,7
8,calc,calc.word,7
9,fit,fit.linear,18


In [9]:
for ty in ("calc.dates", "table.join", "fit.logistic"):
    t = next(t for t in dev if t["type"] == ty)
    print(f"[{ty}]  answer {t['answer']}  tol {t['tol']:g}  files {list(t['files'])}")
    print(t["objective"], "\n")

[calc.dates]  answer 13  tol 1e-06  files []
How many days are there from 15 April 2017 to April 28, 2017? Give a whole number. 

[table.join]  answer 62.5  tol 0.015  files ['energy_use.csv', 'buildings.csv', 'data_dictionary.csv']
You are given daily electricity readings of a company's sites in energy_use.csv, with a data dictionary. What is the median occupancy of the readings that belong to a site with a floor surface of at least 3583 square metres (in people)? energy_use.csv uses ';' to separate columns and a comma as the decimal mark. Missing values are written '-'. Ignore values that are missing or not recorded. energy_use.csv contains duplicated rows (same identifier); keep one copy of each. In energy_use.csv the site key carries the prefix 'SITE-'; buildings.csv does not. buildings.csv uses the same separator and decimal mark. Round your answer to 2 decimal places. 

[fit.logistic]  answer 0.909  tol 0.0015  files ['train.csv', 'test.csv', 'data_dictionary.csv']
train.csv and 

### Score and timing

- **One number per task.** The objective states the rounding: *n decimals* → tolerance
  `1.5 × 10⁻ⁿ`; *an integer* → `1e-6`. Inside it the task scores 1, otherwise 0. An exact,
  unrounded value is always inside.
- **Score = 100 × the mean over all tasks.** The private set has 120 tasks: 45 without files,
  50 tables, 25 fits, with other draws and a few table types that `dev.json` does not have.
- **Time.** `Agent()` has 60 s to load its model, then 15 calls of 8 tasks, **40 s per call**.
  A call that raises or takes longer ends the run and the deployment fails.

In [10]:
for answer in (41.07, "41.07", 41.069, 41.08, "41,07", None):
    print(repr(answer), "->", scoring.score_answer({"answer": 41.07, "tol": 0.015}, answer))

41.07 -> (1.0, None)
'41.07' -> (1.0, None)
41.069 -> (1.0, None)
41.08 -> (1.0, None)
'41,07' -> (0.0, "not a plain number: '41,07'")
None -> (0.0, 'expected a number, got NoneType')


---

## 2. Stage 1: the model, a structured answer, a parser

### One call

`dsh.load_llm` loads a model of the platform's list; `chat` takes a list of conversations and
generates all the replies in one batch (much cheaper than one call per task).

In [11]:
llm = dsh.load_llm("Qwen/Qwen2.5-1.5B-Instruct")
reply = llm.chat([[{"role": "user", "content": no_file["objective"]}]], max_new_tokens=300)[0]
print(no_file["objective"], "\n\n---\n", reply, "\n\ngold:", no_file["answer"])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Compute 88 × 32. The answer is an integer. 

---
 To compute \( 88 \times 32 \), we can use the distributive property of multiplication over addition:

\[ 88 \times 32 = (80 + 8) \times 32 \]

Now, distribute \( 32 \):

\[ 80 \times 32 + 8 \times 32 \]

First, calculate \( 80 \times 32 \):

\[ 80 \times 32 = 2560 \]

Next, calculate \( 8 \times 32 \):

\[ 8 \times 32 = 256 \]

Add these two results together:

\[ 2560 + 256 = 2816 \]

So, \( 88 \times 32 = 2816 \). 

gold: 2816


The reply is prose. The scorer wants one number. Stage 1 is three decisions: **what the
prompt asks for**, **how the answer is marked**, and **how your code reads it back**. We measure
them on 24 tasks without files (the cell takes a few minutes on a T4).

In [12]:
calc = [t for t in dev if not t["files"]]
sample = calc[:24]

def measure(name, system, parse, max_new_tokens=512):
    convs = [[{"role": "system", "content": system}, {"role": "user", "content": t["objective"]}]
             for t in sample]
    t0 = time.time()
    replies = llm.chat(convs, max_new_tokens=max_new_tokens)
    secs = time.time() - t0
    answers = [parse(r) for r in replies]
    score = sum(scoring.score_answer(t, a if a is not None else 0)[0] for t, a in zip(sample, answers))
    print(f"{name:34s} {100 * score / len(sample):5.1f} / 100   {secs / len(sample):.2f} s per task")
    return replies

**An example last line.** Show the expected last line with an example value and read
the number after `####`.

In [14]:
copy = measure("example last line '#### 42.5'",
               "Answer the question. End with a line like: #### 42.5",
               lambda r: dsh.last_number(r.split("####")[-1]))
print(copy[0][:300])

example last line '#### 42.5'        8.3 / 100   2.51 s per task
The product of 88 and 32 is 2816, which is indeed an integer.

#### 2816


**Three designs.** Reason first, then mark the answer:

1. a last line `ANSWER: <number>`, read by a strict parser — or with a **fallback** to the last
   number written;
2. a JSON object `{"reasoning": ..., "answer": ...}`;
3. two calls: reason freely, then ask for the number only.

In [15]:
TAG = ("Solve the problem. Reason step by step, writing each calculation. "
       "Then write a last line of the form ANSWER: <number>, with the number only.")
tag = measure("ANSWER line, strict", TAG, dsh.tagged_number)
_ = measure("ANSWER line, else last number", TAG, lambda r: dsh.tagged_number(r) or dsh.last_number(r))

ANSWER line, strict                 25.0 / 100   2.61 s per task
ANSWER line, else last number       29.2 / 100   2.49 s per task


In [16]:
def from_json(r):
    try:
        return dsh.to_number(json.loads(re.sub(r"^```(json)?|```$", "", r.strip()).strip())["answer"])
    except Exception:
        return dsh.last_number(r)

_ = measure("JSON {reasoning, answer}",
            'Reply with JSON only: {"reasoning": "<short working>", "answer": <number>}', from_json)

JSON {reasoning, answer}            12.5 / 100   1.17 s per task


In [17]:
convs = [[{"role": "system", "content": "Solve the problem. Reason step by step."},
          {"role": "user", "content": t["objective"]}] for t in sample]
first = llm.chat(convs, max_new_tokens=512)
second = llm.chat([c + [{"role": "assistant", "content": r},
                        {"role": "user", "content": "Write only the final answer as a number, rounded as asked."}]
                   for c, r in zip(convs, first)], max_new_tokens=16)
score = sum(scoring.score_answer(t, dsh.last_number(r) or 0)[0] for t, r in zip(sample, second))
print(f"two calls: {100 * score / len(sample):.1f} / 100")

two calls: 29.2 / 100


Measured on the 65 dev tasks without files on the platform's GPU (Qwen2.5-1.5B):


| design | score (/100) | format obeyed |
|---|---|---|
| example last line '#### 42.5' | 23.1 | 14/65 |
| ANSWER line, strict parser | 15.4 | 18/65 |
| ANSWER line, else the last number | 26.2 | 18/65 |
| JSON {reasoning, answer}, else the last number | 13.8 | 54/65 |
| two calls: reason, then the number only | 32.3 | 19/65 |

- **the parser is a design decision**: the same replies score 15.4 with a strict `ANSWER:`
  parser and 26.2 with a fallback to the last number;
- **the best-obeyed format is not the most accurate**: JSON is followed most often and scores
  least — writing JSON takes the place of the reasoning;
- **separating the reasoning from the answer pays**: two calls score best, for one more short
  generation.

`stage1_direct.py` uses the `ANSWER:` line with the fallback: simple, one call. Two calls is
your first measured improvement.

### Where it fails, by type

In [18]:
answers = [dsh.tagged_number(r) or dsh.last_number(r) for r in tag]
by_type = collections.defaultdict(list)
for t, a in zip(sample, answers):
    by_type[t["type"]].append(scoring.score_answer(t, a or 0)[0])
for ty, v in sorted(by_type.items()):
    print(f"{ty:20s} {100 * sum(v) / len(v):5.1f}  ({len(v)} tasks)")

calc.arithmetic      100.0  (2 tasks)
calc.dates            33.3  (3 tasks)
calc.functions         0.0  (3 tasks)
calc.integer         100.0  (1 tasks)
calc.percent           0.0  (2 tasks)
calc.probability       0.0  (5 tasks)
calc.stats_inline     50.0  (4 tasks)
calc.word             25.0  (4 tasks)


Products of large numbers, standard deviations, compound interest, logarithms, days of the
week: the model reasons correctly and computes wrongly. That is what a tool fixes (section 3).

### Never let a task end the run

One exception inside `solve` ends the run and fails the deployment. `stage1_direct.py` wraps the
generation in `try/except` and answers `0` for a task it cannot read: a wrong answer costs one
task, an exception costs all of them.

### With files: the wall

`stage1_direct.py` pastes the first lines of each file into the prompt. Run it on the platform's
test run (16 dev tasks, 2 calls) and on the table types. A T4 is about three times slower than the
platform's RTX 4090, so give each call 120 s here instead of 40.

In [19]:
!python localtest.py stage1_direct.py --test-run --timeout 120

Loading weights: 100% 338/338 [00:05<00:00, 59.80it/s]
Agent() loaded in 17.2 s

16 tasks in 2 calls, 49 s (24.5 s per call, limit 120 s)

score by type
  calc.arithmetic       100.0   (1 tasks)
  calc.dates              0.0   (1 tasks)
  calc.functions          0.0   (1 tasks)
  calc.growth           100.0   (1 tasks)
  calc.integer          100.0   (1 tasks)
  calc.percent            0.0   (1 tasks)
  calc.probability        0.0   (1 tasks)
  calc.stats_inline       0.0   (1 tasks)
  calc.word               0.0   (1 tasks)
  fit.linear              0.0   (1 tasks)
  fit.logistic            0.0   (1 tasks)
  table.derive            0.0   (1 tasks)
  table.filter            0.0   (1 tasks)
  table.group             0.0   (1 tasks)
  table.join              0.0   (1 tasks)
  table.stat            100.0   (1 tasks)

score 25.0 | no files 33.3, tables 20.0, fits 0.0 | 16/16 answered, 0 format errors
SCORE 25.0


In [20]:
!python localtest.py stage1_direct.py --type table.stat --type table.filter --limit 16 --timeout 120

Loading weights: 100% 338/338 [00:02<00:00, 130.36it/s]
Agent() loaded in 12.3 s

16 tasks in 2 calls, 44 s (22.1 s per call, limit 120 s)

score by type
  table.filter            0.0   (7 tasks)
  table.stat             11.1   (9 tasks)

score 6.2 | no files 0.0, tables 6.2, fits 0.0 | 16/16 answered, 0 format errors
SCORE 6.2


The model never sees the data: a statistic over 300 rows cannot come from 6 lines. Measured on the
full `dev.json` on the platform's GPU, stage 1 scores **10.6** (no files 26.2,
tables 2.5, fits 0.0). `stage1_direct.py` is your first submission
(section 6).

---

## 3. Stage 2: a tool and a loop (a skeleton)

A tool is a function your code runs on the model's behalf. The model asks for it in a fixed
format, your code runs it and sends the result back, and the model continues. Six parts:

| part | in `stage2_tool_loop.py` |
|---|---|
| tool spec | the system prompt says what the calculator accepts and how to call it |
| call format | `<call tool="calculator">EXPRESSION</call>`, shown once in a worked exchange (`DEMO`); generation stops at `</call>` |
| call parser | `dsh.parse_tool_call(reply)` → `("calculator", "15.18 - 15.9")` or `None` |
| execution | `observe(call)`: `dsh.calculator`, an error returned as text, never raised |
| observation | a user message `Result: -0.72` (or `Error: ...`) appended to the conversation |
| stop rule | a reply without a call, `MAX_ROUNDS` rounds, or the clock |

The worked exchange matters: with the format only described in the system prompt, the 1.5B model
writes its own (`CALL: ...`) and guesses the result; shown once, it calls the tool (measured on the
65 tasks without files: 9.2 without the exchange, 41.5 with it).

One round, by hand:

In [36]:
import importlib, stage2_tool_loop as s2
import math

importlib.reload(s2)
t = next(t for t in dev if t["type"] == "calc.functions")
conv = [{"role": "system", "content": s2.SYSTEM}] + s2.DEMO + [{"role": "user", "content": t["objective"]}]
r = llm.chat([conv], max_new_tokens=256, stop=["</call>"])[0]
print(r)
call = dsh.parse_tool_call(r)
print("\ncall:", call, "\nobservation:", s2.observe(call) if call else None)

To find the base-10 logarithm of 2804, we can use the `log` function with base 10.
<call tool="calculator">math.log(2804, 10)</call>


call: ('calculator', 'math.log(2804, 10)') 
observation: Result: 3.447778009


In [37]:
if call:
    conv += [{"role": "assistant", "content": r}, {"role": "user", "content": s2.observe(call)}]
    print(llm.chat([conv], max_new_tokens=256, stop=["</call>"])[0])
print("\ngold:", t["answer"])

The base-10 logarithm of 2804 is approximately 3.4478 when rounded to four decimal places.
ANSWER: 3.4478

gold: 3.4478


**Your turn.** Write the loop body marked `TODO` in `stage2_tool_loop.py` (about ten lines:
keep the reply, parse a call, append the reply and the observation, keep the conversation
active). Then compare it with stage 1 on the computation types:

In [38]:
CALC = " ".join(f"--type {ty}" for ty in sorted({t["type"] for t in dev if not t["files"]}))
!python localtest.py stage1_direct.py {CALC} --timeout 120 | tail -14
!python localtest.py stage2_tool_loop.py {CALC} --timeout 120 | tail -14

Loading weights: 100% 338/338 [00:02<00:00, 130.36it/s]

score by type
  calc.arithmetic        37.5   (8 tasks)
  calc.dates             37.5   (8 tasks)
  calc.functions         14.3   (7 tasks)
  calc.growth            28.6   (7 tasks)
  calc.integer           14.3   (7 tasks)
  calc.percent           14.3   (7 tasks)
  calc.probability       14.3   (7 tasks)
  calc.stats_inline      28.6   (7 tasks)
  calc.word              28.6   (7 tasks)

score 24.6 | no files 24.6, tables 0.0, fits 0.0 | 65/65 answered, 0 format errors
SCORE 24.6
Loading weights: 100% 338/338 [00:02<00:00, 130.57it/s]

score by type
  calc.arithmetic        50.0   (8 tasks)
  calc.dates             25.0   (8 tasks)
  calc.functions         85.7   (7 tasks)
  calc.growth            71.4   (7 tasks)
  calc.integer           14.3   (7 tasks)
  calc.percent           71.4   (7 tasks)
  calc.probability        0.0   (7 tasks)
  calc.stats_inline      42.9   (7 tasks)
  calc.word              42.9   (7 tasks)

score 

On the platform's GPU, the solved loop takes the tasks without files from 26.2 to
35.4 with the same model. It does nothing for the tables and the fits: the model still
never reads the data.

---

## 4. Directions

No solution here: these are the design decisions your project is about.

- **Routing.** Recognise the kind of task from the objective (files or not, `train.csv`, words like
  *percentile*, *join*, *probability*) and send it to the right prompt and tools. The private set
  has table types `dev.json` does not: a router keyed on dev wordings fails there.
- **Read the files once, yourself.** Parse each CSV with the rules the objective states, link each
  column to its dictionary description, and give the model a compact, clean view: names,
  descriptions, units, types, a few values. The model then reasons about meaning, not about bytes.
- **Data tools or code execution.** Either a few tools with arguments (`stat(column, filter, ...)`),
  easy to call and to check, or the model writes pandas code that `dsh.run_python` runs
  (general, but a 1.5B model's code fails often). Measure both.
- **A fitting tool for `fit.*`.** scikit-learn is on the platform: `LinearRegression()`, and
  `LogisticRegression(C=np.inf)` for an unregularised fit (its default is regularised, and the
  tasks are not). A tool the model calls with the target, the features and the rows beats code it
  writes from scratch.
- **Checks, repair, pacing.** Is the number in a plausible range, rounded as asked, an integer when
  one is asked? Send failed code back once with its error. Two batched model rounds fit in 40 s;
  a third may not.
- **The model.** Five are mounted: `Qwen2.5-0.5B/1.5B-Instruct`, `Qwen2.5-Coder-1.5B-Instruct`,
  `DeepSeek-R1-Distill-Qwen-1.5B`, `Qwen3-1.7B`. Change it last, and measure.

Measured on `dev.json` (platform GPU), the ladder these directions lead to:

| system | model | score | no files | tables | fits |
|---|---|---|---|---|---|
| stage 1: answer marked `ANSWER:`, parsed with a fallback | Qwen2.5-1.5B | 10.6 | 26.2 | 2.5 | 0.0 |
| stage 1 | Qwen3-1.7B | 16.7 | 44.6 | 0.0 | 2.9 |
| stage 2: + a calculator in a loop | Qwen2.5-1.5B | 13.3 | 35.4 | 1.2 | 0.0 |
| stage 2 | Qwen3-1.7B | 22.2 | 60.0 | 1.2 | 0.0 |
| stage 2 with routing: the loop without files, stage 1 with files | Qwen2.5-1.5B | 13.3 | 35.4 | 1.2 | 0.0 |
| reference: files read by the system, data view, code, one repair | Qwen2.5-1.5B | 46.1 | 86.2 | 28.8 | 11.4 |
| reference | Qwen2.5-Coder-1.5B | 53.3 | 81.5 | 47.5 | 14.3 |
| reference | Qwen3-1.7B | 62.2 | 80.0 | 62.5 | 28.6 |

---

## 5. Measure

`localtest.py --out run.json` writes every task's answer, gold, score and error. Group the
failures before you change anything.

In [39]:
!python localtest.py stage1_direct.py --test-run --timeout 120 --out run.json > /dev/null
run = json.load(open("run.json"))
d = pd.DataFrame(run["details"])
print(d.groupby("type")["score"].mean().mul(100).round(1))
d[d.score == 0][["id", "type", "answer", "gold", "error"]].head(10)

Loading weights: 100% 338/338 [00:02<00:00, 135.87it/s]
type
calc.arithmetic      100.0
calc.dates             0.0
calc.functions         0.0
calc.growth          100.0
calc.integer         100.0
calc.percent           0.0
calc.probability       0.0
calc.stats_inline      0.0
calc.word              0.0
fit.linear             0.0
fit.logistic           0.0
table.derive           0.0
table.filter           0.0
table.group            0.0
table.join             0.0
table.stat           100.0
Name: score, dtype: float64


,id,type,answer,gold,error
0,d_002,fit.logistic,4.0000,0.9090,None
2,d_019,calc.probability,4.3300,4.1200,None
3,d_026,calc.dates,28.0000,13.0000,None
5,d_005,table.join,129.5000,62.5000,None
6,d_014,calc.word,10.0000,10.5900,None
8,d_006,table.group,30.0000,30.7300,None
10,d_020,calc.functions,3.4457,3.4478,None
11,d_010,calc.percent,854.7900,851.9500,None
12,d_015,table.derive,2.0000,0.1000,None
13,d_001,table.filter,1.0000,23.5000,None


- **A ladder.** Keep one table: system version, score by family, seconds per call. One change
  per line.
- **By type.** `dev.json` has 7 to 18 tasks per type: a difference of one task is 6–14 points on a
  type. Look at families and at the whole score.
- **Error causes.** Wrong reading of a file? Wrong column? Right code, wrong rounding? A format
  error? Each cause has its own fix.
- **Your own validation set.** `dev.json` is small, and the private set has other draws: write a
  generator for the types you work on (a few lines of Python with its own gold) and measure on
  hundreds of tasks.
- **Time.** Measure seconds per call on the platform (the run's log) and keep a margin: the T4 is
  slower, the RTX 4090 faster, a long objective slower.

---

## 6. Submit

Upload your agent file **as `agent.py`**, with `dsh.py`, and choose the **PyTorch** runtime. A
deployment runs the test run (16 dev tasks), then the scored run (120 private tasks).

In [ ]:
import shutil
shutil.copy("stage1_direct.py", "agent.py")
result = client.submit(challenge_id=CHALLENGE_ID, files=["agent.py", "dsh.py"],
                       submission_name="stage 1",
                       runtime={"language": "python", "framework": "torch"},
                       wait=True, timeout_sec=1800)
print(result["status"]["status"], result["status"]["last_status_message"])

The leaderboard shows the score and its three family means. Deployments are limited per day:
test locally first, submit when a change is measured.